# Aula 05 - Aprendizado Supervisionado parte I

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**24/08/2026 - Sprint 2 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula05.ipynb)

## O que este notebook é

É o **primeiro modelo preditivo do case**, o Modelo 1 do TAPI: uma regressão da produção
trimestral de frango no Brasil. A Aula 04 entregou a base analítica pronta; aqui ela vira um
modelo treinado, avaliado em dados que ele não viu e comparado contra a forma como a LDC projeta
hoje.

O protocolo de avaliação usado aqui (corte de oito trimestres, baseline de coeficiente fixo,
repetição em doze janelas) está registrado em `docs/adrs/ADR-008`, e as Aulas 07, 10 e 11 vão
comparar os resultados delas contra os números medidos aqui.

## Ao final deste notebook você terá

1. ajustado uma `LinearRegression` sobre a base de frango, com `fit` e `predict`;
2. separado treino e teste **por corte de data**, e medido o que muda ao sortear;
3. calculado RMSE e MAPE do seu modelo e de três baselines diferentes;
4. decidido, com número, se `producao_leite` entra ou fica fora das entradas;
5. testado a normalidade dos **resíduos**, que é onde a suposição de mínimos quadrados recai;
6. repetido a avaliação em doze janelas, para saber se a vantagem do modelo se sustenta.

## 1. Onde estão os arquivos

Mesma resolução de caminho das aulas anteriores: funciona no repositório clonado (onde os CSVs
estão em `../dados/`) e no Colab (onde são baixados da versão publicada do repositório).

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"

BASE_LOCAL = os.path.join("..", "dados")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            urllib.request.urlretrieve(BASE_BRUTA + arquivo, arquivo)
        caminhos[nome] = arquivo

for nome, caminho in caminhos.items():
    print("%-16s -> %s" % (nome, caminho))

## 2. A base analítica da Aula 04, reconstruída

Nada foi gravado em disco na aula passada, de propósito: a base é reconstruída a partir dos CSVs
crus a cada execução. A célula abaixo repete, em uma função, o que a Aula 04 fez em nove seções.

Se alguma linha aqui não fizer sentido, o lugar de conferir é `notebooks/aula04.ipynb`, que
constrói cada uma delas passo a passo.

In [ ]:
def montar_base():
    """Reproduz a base analitica da Aula 04 a partir dos cinco CSVs crus."""
    base = None
    for nome in SERIES:
        coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
                  .rename(columns={"valor": nome}))
        base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
    base = base.sort_values("periodo").reset_index(drop=True)

    base["ano"] = base["periodo"].str[:4].astype(int)
    base["trimestre"] = base["periodo"].str[-1].astype(int)

    # defasagens: o passado entra na linha do presente (Aula 04, secao 4)
    base["frangos_lag1"] = base[ALVO].shift(1)
    base["frangos_lag4"] = base[ALVO].shift(4)

    # sazonalidade como par seno/cosseno (Aula 04, secao 5)
    base["sen"] = np.sin(2 * np.pi * base["trimestre"] / 4)
    base["cos"] = np.cos(2 * np.pi * base["trimestre"] / 4)

    # as outras quatro series tambem defasadas, para a secao 6
    for nome in SERIES:
        if nome != ALVO:
            base[nome + "_lag1"] = base[nome].shift(1)

    return base.dropna().reset_index(drop=True)


analitica = montar_base()
print("base analitica:", analitica.shape)
print("de", analitica["periodo"].iloc[0], "ate", analitica["periodo"].iloc[-1])
print()
print(analitica[["periodo", ALVO, "frangos_lag1", "frangos_lag4", "sen", "cos"]]
      .head(4).to_string(index=False))

## 3. O primeiro modelo

`LinearRegression` procura os coeficientes que minimizam a soma dos quadrados dos erros. A
interface é a mesma de todo estimador do scikit-learn, e ela não muda nas Aulas 06 a 11:

- `fit(X, y)` aprende;
- `predict(X)` aplica.

Duas exigências de forma: `X` é sempre bidimensional (uma linha por observação, uma coluna por
característica) e `y` é unidimensional. Passar uma `Series` como `X` levanta erro de forma.

In [ ]:
FEATURES = ["frangos_lag1", "frangos_lag4", "sen", "cos"]

X = analitica[FEATURES]
y = analitica[ALVO]

modelo = LinearRegression()
modelo.fit(X, y)

print("R2 no treino:", round(modelo.score(X, y), 4))
print()
for nome, coeficiente in zip(FEATURES, modelo.coef_):
    print("%-16s %+14.6f" % (nome, coeficiente))
print("%-16s %+14.4g" % ("intercepto", modelo.intercept_))
print()
soma = modelo.coef_[0] + modelo.coef_[1]
print("coef(lag1) + coef(lag4) = %.4f" % soma)

Os dois coeficientes de defasagem somam praticamente **1**. O modelo aprendeu a prever cada
trimestre como uma média ponderada do trimestre anterior (peso perto de 0,69) e do mesmo trimestre
do ano anterior (peso perto de 0,30). Ninguém escreveu essa regra: ela saiu dos mínimos quadrados.

**O `R2` acima foi calculado sobre as mesmas linhas usadas no treino.** Ele responde se o modelo
consegue reproduzir o que já viu, que é a pergunta mais fácil que se pode fazer a ele. A pergunta
que a LDC faz é outra, e a próxima seção monta a separação que permite respondê-la.

## 4. Separar treino de teste, por data

O TAPI pede projeção com horizonte de 24 meses, que nesta base são **oito trimestres**. O corte
reserva os oito últimos e treina no restante, com `iloc` sobre a base já ordenada por `periodo`.
Nenhuma função de sorteio participa.

In [ ]:
N_TESTE = 8
corte = len(analitica) - N_TESTE

treino = analitica.iloc[:corte]
teste = analitica.iloc[corte:]

print("treino: %3d linhas  (%s a %s)"
      % (len(treino), treino["periodo"].iloc[0], treino["periodo"].iloc[-1]))
print("teste : %3d linhas  (%s a %s)"
      % (len(teste), teste["periodo"].iloc[0], teste["periodo"].iloc[-1]))

### 4.1 O que o sorteio faria

`train_test_split` tem `shuffle=True` como padrão. A célula abaixo mostra quais períodos ele
manda para o teste nesta base.

In [ ]:
from sklearn.model_selection import train_test_split

_, teste_sorteado = train_test_split(analitica, test_size=N_TESTE, random_state=42)
print("teste sorteado:", sorted(teste_sorteado["periodo"].tolist()))
print("teste por data:", teste["periodo"].tolist())

Os períodos sorteados se espalham por quase trinta anos, e cada um deles tem vizinhos temporais
dentro do treino, dos dois lados. Prever um trimestre cercado de trimestres conhecidos é uma
tarefa mais fácil, e diferente da que a LDC precisa resolver.

### 4.2 Vazamento, medido

O caso extremo dessa mesma ideia é colocar o futuro como coluna de entrada. `shift(-1)` traz o
valor do trimestre seguinte para a linha de hoje.

In [ ]:
def avaliar(features, dados, n_teste=N_TESTE, padronizar=True):
    """Ajusta no treino, preve no teste e devolve RMSE e MAPE dos dois."""
    c = len(dados) - n_teste
    tr, te = dados.iloc[:c], dados.iloc[c:]
    X_tr, y_tr = tr[features].to_numpy(float), tr[ALVO].to_numpy(float)
    X_te, y_te = te[features].to_numpy(float), te[ALVO].to_numpy(float)

    if padronizar:
        # fit SO no treino; o teste apenas recebe transform
        escalador = StandardScaler().fit(X_tr)
        X_tr, X_te = escalador.transform(X_tr), escalador.transform(X_te)

    m = LinearRegression().fit(X_tr, y_tr)
    p_tr, p_te = m.predict(X_tr), m.predict(X_te)
    return {
        "rmse_treino": np.sqrt(mean_squared_error(y_tr, p_tr)),
        "mape_treino": mean_absolute_percentage_error(y_tr, p_tr) * 100,
        "rmse_teste": np.sqrt(mean_squared_error(y_te, p_te)),
        "mape_teste": mean_absolute_percentage_error(y_te, p_te) * 100,
        "modelo": m,
        "previsto": p_te,
        "teste": te,
    }


# base auxiliar SO para a demonstracao: a coluna do futuro nao entra na base oficial
demo = analitica.copy()
demo["frangos_futuro"] = demo[ALVO].shift(-1)
demo = demo.dropna().reset_index(drop=True)

honesto = avaliar(FEATURES, demo)
vazado = avaliar(FEATURES + ["frangos_futuro"], demo)

print("teste da demonstracao: %s a %s"
      % (honesto["teste"]["periodo"].iloc[0], honesto["teste"]["periodo"].iloc[-1]))
print("honesto                    MAPE %.2f%%" % honesto["mape_teste"])
print("com a coluna do futuro     MAPE %.2f%%" % vazado["mape_teste"])

O erro cai cerca de um terço, o código roda sem nenhum aviso e o modelo é inútil em produção,
porque o próximo trimestre ainda não aconteceu no momento da previsão.

**A lição de método:** vazamento se detecta lendo o desenho do experimento. A métrica de um modelo
vazado é justamente a que parece boa, então ela não serve como alarme.

Repare também que `demo` tem uma linha a menos que `analitica`: `shift(-1)` consome a última
linha, e o `dropna()` a remove. Por isso a coluna do futuro foi criada numa cópia, e não na base
oficial. Uma coluna de diagnóstico que muda o tamanho da base muda todas as métricas junto.

## 5. RMSE e MAPE, e contra o que comparar

As duas métricas do módulo, definidas no `PLANO_DE_ENSINO.md`, respondem a perguntas diferentes:

- **RMSE** está em quilogramas, que é a unidade em que a LDC dimensiona estoque. Pune erro grande
  com o quadrado.
- **MAPE** está em porcentagem, e permite comparar o desempenho entre séries de tamanhos
  diferentes. Explode quando o valor real se aproxima de zero.

A comparação é contra o que a LDC já faz. A abordagem de coeficientes estáticos vira, aqui, o
valor do mesmo trimestre do ano anterior multiplicado por um fator constante, **estimado só no
treino**.

In [ ]:
resultado = avaliar(FEATURES, analitica)
y_teste = teste[ALVO].to_numpy(float)

# o fator sai SO do treino: calcular sobre a base inteira deixaria o teste entrar na conta
fator = float((treino[ALVO] / treino["frangos_lag4"]).mean())

baselines = {
    "A. repete o trimestre anterior": teste["frangos_lag1"].to_numpy(float),
    "B. repete o mesmo trimestre do ano anterior": teste["frangos_lag4"].to_numpy(float),
    "C. ano anterior x coeficiente fixo (%.4f)" % fator:
        teste["frangos_lag4"].to_numpy(float) * fator,
}

print("%-46s %14s %8s" % ("previsor", "RMSE (Mkg)", "MAPE"))
for nome, previsao in baselines.items():
    rmse = np.sqrt(mean_squared_error(y_teste, previsao))
    mape = mean_absolute_percentage_error(y_teste, previsao) * 100
    print("%-46s %14.1f %7.2f%%" % (nome, rmse / 1e6, mape))

print("%-46s %14.1f %7.2f%%" % ("regressao linear (lag1, lag4, sen, cos)",
                                resultado["rmse_teste"] / 1e6, resultado["mape_teste"]))

O modelo ganha da melhor baseline por **0,10 ponto percentual** de MAPE. É uma vantagem apertada,
e a seção 8 volta a esse número com uma medição que decide se ele se sustenta.

Repare que a baseline B, sozinha, é a pior das três: repetir o ano anterior sem correção ignora o
crescimento de longo prazo da série. O fator de 1,05 que a baseline C aplica é exatamente essa
correção, e ele sozinho já derruba o erro de 4,5% para 1,7%.

## 6. O leite entra ou fica fora?

A Aula 04 testou a hipótese de correlação nula entre `producao_leite` e `abate_frangos` e
encontrou r = +0,96 sobre o nível e praticamente zero sobre a primeira diferença. A conclusão
registrada foi que o leite fica fora, e volta **apenas se reduzir o erro em dados que o modelo não
viu**. Agora existe como medir isso.

In [ ]:
combinacoes = [
    ("lag1", ["frangos_lag1"]),
    ("lag1 + lag4", ["frangos_lag1", "frangos_lag4"]),
    ("lag1 + lag4 + sen/cos", FEATURES),
    ("+ leite do mesmo trimestre", FEATURES + ["producao_leite"]),
    ("+ leite defasado em 1 trimestre", FEATURES + ["producao_leite_lag1"]),
    ("+ as 4 series defasadas", FEATURES + [s + "_lag1" for s in SERIES if s != ALVO]),
]

print("%-36s %14s %14s" % ("entradas", "MAPE treino", "MAPE teste"))
for nome, feats in combinacoes:
    r = avaliar(feats, analitica)
    print("%-36s %13.2f%% %13.2f%%" % (nome, r["mape_treino"], r["mape_teste"]))

Duas leituras saem desta tabela.

**A coluna do meio melhora quase sempre.** Acrescentar característica dá ao modelo mais liberdade
para se ajustar às 105 linhas que ele já viu, e por isso o erro de treino tende a cair a cada
linha. Ela não decide nada.

**A coluna da direita reprova o leite nas duas formas testadas.** O MAPE de teste sobe, apesar de
o de treino cair. É a definição operacional de overfitting: ganho no treino que não se transfere
para dados novos. A hipótese da Aula 04 se confirma em dados que o modelo não viu.

A última linha mostra que as outras séries **defasadas** melhoram de verdade. Elas não entram no
modelo de hoje, que é o primeiro, e ficam registradas como candidata para a Aula 07.

## 7. A figura da ART.4: histórico contra previsão

A ART.4 (UX parte 2) pede a comunicação visual do produto. A figura abaixo é a versão mínima
defensável dela: o realizado, a previsão do modelo e a da baseline, nos oito trimestres
reservados.

In [ ]:
import matplotlib.pyplot as plt

TINTA, DESTAQUE, APOIO = "#2e2640", "#ff4545", "#89cea5"

previsto = resultado["previsto"]
baseline_c = teste["frangos_lag4"].to_numpy(float) * fator
z = np.arange(N_TESTE)

fig, eixo = plt.subplots(figsize=(11, 4.6))
eixo.plot(z, y_teste / 1e9, color=TINTA, linewidth=2.5, marker="o",
          label="realizado")
eixo.plot(z, previsto / 1e9, color=DESTAQUE, linewidth=2.5, linestyle="--", marker="^",
          label="modelo, MAPE %.2f%%" % resultado["mape_teste"])
eixo.plot(z, baseline_c / 1e9, color=APOIO, linewidth=3, linestyle=":", marker="s",
          label="baseline, MAPE %.2f%%"
                % (mean_absolute_percentage_error(y_teste, baseline_c) * 100))
eixo.set_xticks(z)
eixo.set_xticklabels(teste["periodo"], rotation=45, ha="right")
eixo.set_ylabel("bilhões de kg")
eixo.spines[["top", "right"]].set_visible(False)
eixo.legend(frameon=False)
fig.tight_layout()
plt.show()

erro_relativo = (previsto - y_teste) / y_teste * 100
print("erro do modelo em cada trimestre, em %:")
for periodo, e in zip(teste["periodo"], erro_relativo):
    print("   %s  %+6.2f%%" % (periodo, e))

O modelo **subestima sete dos oito trimestres**, e só supera o realizado em 2024-T4. Um erro
quase sempre no mesmo sentido é informação: significa que o modelo carrega um viés de nível
no período de teste, e não apenas ruído. A causa provável é a tendência de crescimento recente ser mais forte do que a média dos 29
anos que ele viu no treino.

Para a LDC isso é uma informação acionável, e não um defeito fatal: um erro sistemático para baixo
é corrigível, e um erro sem direção não é.

## 8. O que uma única medição não mostra

### 8.1 A vantagem de 0,10 ponto se sustenta?

Oito trimestres são poucos, e uma diferença de 0,10 ponto percentual cabe dentro do que a escolha
do corte explicaria sozinha. A célula abaixo desloca o corte um trimestre por vez, doze vezes, e
mede modelo e baseline em cada janela.

In [ ]:
print("%-10s %10s %12s   %s" % ("janela", "modelo", "baseline C", "vence"))
mapes_modelo, mapes_baseline = [], []
for k in range(12, 0, -1):
    c = len(analitica) - N_TESTE - k + 1
    tr, te = analitica.iloc[:c], analitica.iloc[c:c + N_TESTE]

    esc = StandardScaler().fit(tr[FEATURES].to_numpy(float))
    m = LinearRegression().fit(esc.transform(tr[FEATURES].to_numpy(float)),
                               tr[ALVO].to_numpy(float))
    p = m.predict(esc.transform(te[FEATURES].to_numpy(float)))

    f = float((tr[ALVO] / tr["frangos_lag4"]).mean())
    b = te["frangos_lag4"].to_numpy(float) * f

    yt = te[ALVO].to_numpy(float)
    mm = mean_absolute_percentage_error(yt, p) * 100
    mb = mean_absolute_percentage_error(yt, b) * 100
    mapes_modelo.append(mm)
    mapes_baseline.append(mb)
    print("%-10s %9.2f%% %11.2f%%   %s"
          % (te["periodo"].iloc[0], mm, mb, "modelo" if mm < mb else "baseline"))

vitorias = sum(1 for a, b in zip(mapes_modelo, mapes_baseline) if a < b)
print()
print("modelo vence em %d de %d janelas" % (vitorias, len(mapes_modelo)))
print("MAPE medio: modelo %.2f%%, baseline %.2f%%"
      % (np.mean(mapes_modelo), np.mean(mapes_baseline)))

O modelo vence nas doze janelas, com margem média de quase um ponto percentual. A vantagem
apertada da última janela é o pior caso das doze, e não o resultado típico.

**Ressalva honesta:** as doze janelas se sobrepõem, então essas medidas não são independentes
entre si. Elas reduzem o risco de a conclusão vir de um acaso de calendário, e não equivalem a
doze amostras independentes. A Aula 10 formaliza essa ideia com `TimeSeriesSplit`.

### 8.2 A dívida sobre padronização

A Aula 04 afirmou que regressão linear sem regularização produz as mesmas previsões com ou sem
padronização. Em álgebra exata, correto. A célula abaixo testa a afirmação nesta base.

In [ ]:
sem = avaliar(FEATURES, analitica, padronizar=False)
com = avaliar(FEATURES, analitica, padronizar=True)

print("coeficientes de sen e cos")
print("  sem padronizar:  sen %+.4g   cos %+.4g" % (sem["modelo"].coef_[2], sem["modelo"].coef_[3]))
escalador = StandardScaler().fit(treino[FEATURES].to_numpy(float))
na_escala_original = com["modelo"].coef_ / escalador.scale_
print("  com padronizar:  sen %+.4g   cos %+.4g   (traduzidos para a escala original)"
      % (na_escala_original[2], na_escala_original[3]))
print()
print("MAPE no teste:  sem padronizar %.2f%%   com padronizar %.2f%%"
      % (sem["mape_teste"], com["mape_teste"]))
print()

X_tr = treino[FEATURES].to_numpy(float)
com_intercepto = np.hstack([np.ones((len(X_tr), 1)), X_tr])
padronizado = np.hstack([np.ones((len(X_tr), 1)), escalador.transform(X_tr)])
print("numero de condicao da matriz de treino")
print("  crua:        %.3g" % np.linalg.cond(com_intercepto))
print("  padronizada: %.3g" % np.linalg.cond(padronizado))

Sem padronizar, os coeficientes de `sen` e `cos` saem da ordem de 1e-10, que é **zero na
prática**: as duas colunas de sazonalidade foram silenciosamente descartadas.

A causa é o **número de condição**. As defasagens estão na casa de 1e9 e o par seno/cosseno na
casa de 1, o que dá à matriz uma condição da ordem de 10 bilhões. A decomposição em valores
singulares que resolve os mínimos quadrados trata as direções de menor valor singular como ruído
numérico e as zera. Padronizar derruba a condição para cerca de 14, e as colunas voltam a
participar.

**A afirmação da Aula 04 não estava errada, estava incompleta:** ela vale em aritmética exata, e a
aritmética de ponto flutuante tem um limite que esta base atravessa. O efeito prático aqui é
pequeno (MAPE de 1,71% para 1,60%), e o efeito de método é grande: um coeficiente zerado por corte
numérico tem exatamente a mesma aparência de um coeficiente que o modelo julgou irrelevante.

**Uma segunda medição, sobre a disciplina de `fit` só no treino.** Ajustar o escalador
sobre treino e teste juntos desloca a média dele em mais de 12% de um desvio, e o MAPE sai
idêntico até a nona casa. Regressão linear sem regularização é invariante a transformação
afim das entradas, então esse vazamento é inofensivo **neste** modelo. A disciplina segue
obrigatória porque KNN, SVM, regressão regularizada e PCA não têm essa invariância, e três
deles aparecem entre as Aulas 06 e 08.

### 8.3 Normalidade, agora nos resíduos

A Aula 04 rejeitou a normalidade das cinco séries e registrou a distinção que fecha aqui: a
suposição de normalidade em mínimos quadrados recai sobre os **resíduos** do modelo ajustado, e
serve aos testes de significância dos coeficientes.

In [ ]:
X_treino = escalador.transform(treino[FEATURES].to_numpy(float))
y_treino = treino[ALVO].to_numpy(float)
residuos = y_treino - com["modelo"].predict(X_treino)

W, p = stats.shapiro(residuos)
print("Shapiro-Wilk nos %d residuos de treino" % len(residuos))
print("  W = %.4f   p = %.4f   ->  %s"
      % (W, p, "rejeita normalidade" if p < 0.05 else "NAO rejeita normalidade"))
print("  media %.4g   desvio %.4g" % (residuos.mean(), residuos.std()))
print()

ordem = np.argsort(np.abs(residuos))[-4:][::-1]
print("os quatro maiores erros de treino:")
for i in ordem:
    print("   %s  real %.4g  previsto %.4g  %+.2f%%"
          % (treino["periodo"].iloc[i], y_treino[i], y_treino[i] - residuos[i],
             residuos[i] / y_treino[i] * 100))

O teste rejeita a normalidade. Os quatro maiores erros são todos de superestimação e caem em
trimestres isolados, e é essa cauda que o Shapiro-Wilk detecta.

**Consequência prática:** valor-p de coeficiente desta regressão fica sem garantia. As previsões e
o RMSE continuam válidos, porque nenhum dos dois depende dessa suposição.

Atribuir cada um desses trimestres a um evento específico do setor é **hipótese**, não resultado:
testá-la exigiria uma variável que a base não tem. Registrar a hipótese e a falta de evidência é
mais honesto do que preencher a lacuna com uma explicação plausível.

## 9. Reprodutibilidade

Este notebook não usa `random_state` em lugar nenhum do modelo final, e isso é uma propriedade,
não um esquecimento: o corte é por posição fixa e `LinearRegression` não tem componente aleatório.
A célula abaixo confirma que duas execuções independentes devolvem exatamente o mesmo número.

In [ ]:
a = avaliar(FEATURES, analitica)["mape_teste"]
b = avaliar(FEATURES, montar_base())["mape_teste"]
print("execucao 1: %.10f%%" % a)
print("execucao 2: %.10f%%" % b)
print("identicas:", a == b)

Os modelos das próximas aulas têm sorteio de verdade (árvores, K-means, divisão de validação), e
ali `random_state` passa a ser obrigatório.

Os dois termos do autoestudo da semana não são sinônimos:

- **reprodutibilidade** é obter o mesmo resultado com os mesmos dados e o mesmo código, que é o
  que a célula acima confirma;
- **replicabilidade** é obter a mesma conclusão com dados novos, que é o que a seção 8.1 mediu ao
  repetir a avaliação em doze janelas.

O primeiro é requisito de engenharia. O segundo é o que sustenta uma recomendação ao parceiro.

## 10. Desafio

Responda no código, e os itens 4 e 5 em texto.

1. Monte a base analítica da série **da sua dupla** (não do frango), com as defasagens de 1 e de 4
   trimestres e a codificação de sazonalidade.
2. Separe treino e teste por corte de data, com os últimos 8 trimestres reservados, e ajuste uma
   `LinearRegression` padronizada.
3. Calcule RMSE e MAPE do seu modelo e da baseline de coeficiente fixo daquela série, com o fator
   estimado só no treino.
4. Em `resposta_4`, escreva se você recomendaria o seu modelo à LDC no lugar do processo atual, e
   com base em qual número.
5. Em `resposta_5`, repita a avaliação em doze janelas e diga se a sua resposta ao item 4 muda.

In [ ]:
MINHA_SERIE = "producao_leite"   # troque pela serie da sua dupla

base = montar_base()
base["alvo_lag1"] = base[MINHA_SERIE].shift(1)
base["alvo_lag4"] = base[MINHA_SERIE].shift(4)
minha = base.dropna().reset_index(drop=True)
MINHAS_FEATURES = ["alvo_lag1", "alvo_lag4", "sen", "cos"]

# 2. corte por data
c = len(minha) - N_TESTE
tr, te = minha.iloc[:c], minha.iloc[c:]
print("treino %d (%s a %s) | teste %d (%s a %s)"
      % (len(tr), tr["periodo"].iloc[0], tr["periodo"].iloc[-1],
         len(te), te["periodo"].iloc[0], te["periodo"].iloc[-1]))

esc = StandardScaler().fit(tr[MINHAS_FEATURES].to_numpy(float))
mod = LinearRegression().fit(esc.transform(tr[MINHAS_FEATURES].to_numpy(float)),
                             tr[MINHA_SERIE].to_numpy(float))
pred = mod.predict(esc.transform(te[MINHAS_FEATURES].to_numpy(float)))

# 3. metricas do modelo e da baseline
yt = te[MINHA_SERIE].to_numpy(float)
f = float((tr[MINHA_SERIE] / tr["alvo_lag4"]).mean())
bl = te["alvo_lag4"].to_numpy(float) * f

for nome, previsao in [("modelo", pred), ("baseline (fator %.4f)" % f, bl)]:
    print("%-26s RMSE %12.4g   MAPE %6.2f%%"
          % (nome, np.sqrt(mean_squared_error(yt, previsao)),
             mean_absolute_percentage_error(yt, previsao) * 100))

resposta_4 = "..."
resposta_5 = "..."